# Ablation Study: Multi-Modal Fusion for Automatic Modulation Classification
### Kaggle Edition

## Experimental Design

| Config | IQ | Constellation | Spectrogram |
|--------|----|---------------|-------------|
| IQ only        | ✓ |   |   |
| Const only     |   | ✓ |   |
| Spec only      |   |   | ✓ |
| IQ + Const     | ✓ | ✓ |   |
| IQ + Spec      | ✓ |   | ✓ |
| Const + Spec   |   | ✓ | ✓ |
| Full Fusion    | ✓ | ✓ | ✓ |

- **5 independent runs** per configuration (seeds 0–4)
- **100 epochs** with `ReduceLROnPlateau` scheduler
- **Auto-resume** — if the session is interrupted, re-running any config cell
  automatically skips already-completed seeds and continues from where it stopped
- All figures saved to `figures/`, results exported to `results/`


## 1. Kaggle Setup & GPU Verification

In [ ]:
import os

# ── Set working directory to Kaggle's persistent output folder ────────────────
os.chdir("/kaggle/working")

# ── Create output directories ──────────────────────────────────────────────────
for d in ["saved_models", "figures", "results"]:
    os.makedirs(d, exist_ok=True)

print("Working directory :", os.getcwd())
print("Output dirs ready :", os.listdir("."))


In [ ]:
# ── Verify GPU is available ───────────────────────────────────────────────────
import subprocess
print(subprocess.getoutput("nvidia-smi"))

import torch
print(f"\nPyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
    print(f"VRAM             : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected — make sure Accelerator is set to GPU in Settings")


## 2. Install & Import Dependencies

In [ ]:
!pip install scipy scikit-learn tqdm -q   # torch & numpy already on Kaggle

import os, time, random, json, csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.signal import spectrogram as scipy_spectrogram
from scipy.ndimage import zoom
from scipy.stats import ttest_ind
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import itertools

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 3. Signal Generation & Dataset

In [ ]:
def add_awgn(signal, snr_db):
    snr_lin = 10 ** (snr_db / 10.0)
    pwr     = np.mean(np.abs(signal) ** 2)
    noise   = np.sqrt(pwr / snr_lin / 2) * (
        np.random.randn(*signal.shape) + 1j * np.random.randn(*signal.shape))
    return signal + noise

def gen_bpsk(n):   return np.random.choice([-1, 1], n).astype(complex)
def gen_qpsk(n):   r=np.random.choice([-1,1],n); i=np.random.choice([-1,1],n); return (r+1j*i)/np.sqrt(2)
def gen_8psk(n):   return np.exp(1j * np.random.randint(0,8,n) * np.pi/4)
def gen_16qam(n):  v=np.array([-3,-1,1,3]);          return (np.random.choice(v,n)+1j*np.random.choice(v,n))/np.sqrt(10)
def gen_64qam(n):  v=np.array([-7,-5,-3,-1,1,3,5,7]); return (np.random.choice(v,n)+1j*np.random.choice(v,n))/np.sqrt(42)

MODULATIONS = {"BPSK": gen_bpsk, "QPSK": gen_qpsk, "8PSK": gen_8psk,
               "16QAM": gen_16qam, "64QAM": gen_64qam}
MOD_NAMES = list(MODULATIONS.keys())

def generate_dataset(samples_per_mod_snr=500, num_symbols=128,
                     snrs=list(range(-20, 22, 2))):
    X, y, snr_arr = [], [], []
    for label, (name, func) in enumerate(MODULATIONS.items()):
        for snr in snrs:
            for _ in range(samples_per_mod_snr):
                noisy = add_awgn(func(num_symbols), snr)
                X.append(np.stack([np.real(noisy), np.imag(noisy)], axis=0))
                y.append(label); snr_arr.append(snr)
    return (np.array(X, dtype=np.float32),
            np.array(y, dtype=np.int64),
            np.array(snr_arr))

SAMPLES_PER_MOD_SNR = 500
NUM_SYMBOLS         = 128
SNRs                = list(range(-20, 22, 2))

print("Generating dataset …")
X, y, snr_vals = generate_dataset(SAMPLES_PER_MOD_SNR, NUM_SYMBOLS, SNRs)
print(f"  Shape  : {X.shape}")
print(f"  Total  : {len(X):,} samples")
print(f"  Classes: {MOD_NAMES}")
print(f"  SNR    : {SNRs[0]} → {SNRs[-1]} dB  ({len(SNRs)} steps)")


## 4. Image Generation & Dataset Class

In [ ]:
IMG_SIZE = 64

def iq_to_constellation(iq, img_size=IMG_SIZE):
    """2-D histogram constellation diagram → (3, H, W) float32 tensor."""
    i, q = iq[0].numpy(), iq[1].numpy()
    hist, _, _ = np.histogram2d(i, q, bins=img_size, range=[[-2,2],[-2,2]])
    hist = hist.astype(np.float32)
    if hist.max() > 0: hist /= hist.max()
    return torch.from_numpy(np.stack([hist, hist, hist]))

def iq_to_spectrogram(iq, img_size=IMG_SIZE, nperseg=64, noverlap=32):
    """Magnitude spectrogram via scipy → (3, H, W) float32 tensor."""
    cx = iq[0].numpy() + 1j * iq[1].numpy()
    _, _, Sxx = scipy_spectrogram(cx, fs=1.0, nperseg=nperseg,
                                  noverlap=noverlap, mode="magnitude")
    Sxx_db   = 10 * np.log10(np.abs(Sxx) + 1e-10)
    Sxx_norm = (Sxx_db - Sxx_db.min()) / (Sxx_db.max() - Sxx_db.min() + 1e-8)
    zf       = (img_size / Sxx_norm.shape[0], img_size / Sxx_norm.shape[1])
    resized  = zoom(Sxx_norm.astype(np.float32), zf, order=1)
    return torch.from_numpy(np.stack([resized, resized, resized]))


class MultiModalDataset(Dataset):
    """Pre-computes constellation and spectrogram images at construction time."""
    def __init__(self, iq_tensors, labels):
        self.iq     = iq_tensors
        self.labels = torch.as_tensor(labels, dtype=torch.long)
        print("  Pre-computing images …")
        consts, specs = [], []
        for i in tqdm(range(len(iq_tensors))):
            consts.append(iq_to_constellation(iq_tensors[i]))
            specs.append(iq_to_spectrogram(iq_tensors[i]))
        self.const_imgs = torch.stack(consts)
        self.spec_imgs  = torch.stack(specs)

    def __len__(self): return len(self.iq)

    def __getitem__(self, idx):
        return (self.iq[idx], self.const_imgs[idx],
                self.spec_imgs[idx], self.labels[idx])


In [ ]:
X_train, X_test, y_train, y_test, snr_train, snr_test = train_test_split(
    X, y, snr_vals, test_size=0.3, random_state=42, stratify=y)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)

print("Building train dataset …")
train_ds = MultiModalDataset(X_train_t, y_train)
print("\nBuilding test dataset …")
test_ds  = MultiModalDataset(X_test_t,  y_test)
print(f"\nTrain : {len(train_ds):,}  |  Test : {len(test_ds):,}")


## 5. Model Architecture

In [ ]:
class IQEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(2,   64,  3, padding=1), nn.BatchNorm1d(64),  nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64,  128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(128, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2),
            nn.AdaptiveAvgPool1d(1))
        self.fc = nn.Linear(256, out_dim)
    def forward(self, x): return self.fc(self.net(x).squeeze(-1))


class ImageEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,  32,  3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1))
        self.fc = nn.Linear(128, out_dim)
    def forward(self, x): return self.fc(self.net(x).squeeze(-1).squeeze(-1))


class FusionModel(nn.Module):
    """
    Flexible multi-modal fusion model.
    Active branches controlled by use_iq / use_const / use_spec.
    Fusion head input dimension adjusts automatically.
    """
    def __init__(self, num_classes=5, feat_dim=128,
                 use_iq=True, use_const=True, use_spec=True):
        super().__init__()
        assert any([use_iq, use_const, use_spec]), "At least one branch required."
        self.use_iq = use_iq; self.use_const = use_const; self.use_spec = use_spec

        if use_iq:    self.iq_enc    = IQEncoder(out_dim=feat_dim)
        if use_const: self.const_enc = ImageEncoder(out_dim=feat_dim)
        if use_spec:  self.spec_enc  = ImageEncoder(out_dim=feat_dim)

        n = sum([use_iq, use_const, use_spec])
        self.fusion = nn.Sequential(
            nn.Linear(feat_dim * n, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes))

    def forward(self, iq, const, spec):
        feats = []
        if self.use_iq:    feats.append(self.iq_enc(iq))
        if self.use_const: feats.append(self.const_enc(const))
        if self.use_spec:  feats.append(self.spec_enc(spec))
        return self.fusion(torch.cat(feats, dim=1))


## 6. Training & Evaluation Helpers

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
NUM_EPOCHS = 100
BATCH_SIZE = 256
NUM_RUNS   = 5
SEEDS      = list(range(NUM_RUNS))
LR         = 1e-3

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Master results store — populated by each config cell below
results = {}

print(f"Epochs : {NUM_EPOCHS}  |  Seeds : {NUM_RUNS}  |  Batch : {BATCH_SIZE}")
print(f"Device : {device}")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def ckpt_path(cfg_name, seed):
    """Return checkpoint path for a given config + seed."""
    safe = cfg_name.lower().replace(" ", "_").replace("+", "plus")
    return os.path.join("saved_models", f"{safe}_seed{seed}.pt")


def all_seeds_done(cfg_name):
    """Return list of seeds already completed for this config."""
    return [s for s in SEEDS if os.path.exists(ckpt_path(cfg_name, s))]


def train_one_run(model, num_epochs, lr):
    opt   = optim.Adam(model.parameters(), lr=lr)
    sched = optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="max", patience=7, factor=0.5)
    crit  = nn.CrossEntropyLoss()

    train_losses, val_accs = [], []
    best_acc, best_state, best_snr_acc = 0.0, None, {}

    for epoch in range(num_epochs):
        # ── Train ──────────────────────────────────────────────────────────
        model.train()
        total_loss = 0
        for iq, const, spec, by in train_loader:
            iq, const, spec, by = (iq.to(device), const.to(device),
                                   spec.to(device), by.to(device))
            opt.zero_grad()
            loss = crit(model(iq, const, spec), by)
            loss.backward(); opt.step()
            total_loss += loss.item()
        train_losses.append(total_loss / len(train_loader))

        # ── Validate ───────────────────────────────────────────────────────
        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for iq, const, spec, by in test_loader:
                iq, const, spec = iq.to(device), const.to(device), spec.to(device)
                preds.extend(model(iq, const, spec).argmax(1).cpu().numpy())
                labels.extend(by.numpy())
        preds, labels = np.array(preds), np.array(labels)
        acc = accuracy_score(labels, preds)
        val_accs.append(acc); sched.step(acc)

        if acc > best_acc:
            best_acc   = acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_snr_acc = {snr: accuracy_score(labels[snr_test == snr],
                                                 preds[snr_test == snr])
                            for snr in np.unique(snr_test)}

    return train_losses, val_accs, best_state, best_snr_acc


def save_fig(fig, filename):
    path = os.path.join("figures", filename)
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print(f"  Figure → {path}")


def plot_learning_curves(cfg_name, all_train_losses, all_val_accs, save=True):
    palette = ["#e15759", "#4e79a7", "#59a14f", "#f28e2b", "#b07aa1"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for run_i, (losses, accs) in enumerate(zip(all_train_losses, all_val_accs)):
        ep = range(1, len(losses) + 1)
        axes[0].plot(ep, losses,               color=palette[run_i], label=f"Seed {run_i}", linewidth=1.5)
        axes[1].plot(ep, [a*100 for a in accs], color=palette[run_i], label=f"Seed {run_i}", linewidth=1.5)
    for ax in axes:
        ax.set_xlabel("Epoch"); ax.legend(fontsize=8.5); ax.grid(True, alpha=0.4)
    axes[0].set_ylabel("Cross-Entropy Loss");    axes[0].set_title(f"{cfg_name} — Training Loss")
    axes[1].set_ylabel("Validation Accuracy (%)"); axes[1].set_title(f"{cfg_name} — Validation Accuracy")
    plt.suptitle(f"Learning Curves: {cfg_name}", fontsize=12, y=1.01)
    plt.tight_layout()
    if save:
        safe = cfg_name.lower().replace(" ", "_").replace("+", "plus")
        save_fig(fig, f"learning_curves_{safe}.png")
    plt.show()


def run_config(cfg_name, use_iq, use_const, use_spec):
    """
    Train NUM_RUNS seeds for one configuration.

    AUTO-RESUME: If a checkpoint already exists for a seed, that seed is
    loaded from disk and skipped — training continues from the next incomplete seed.
    Re-run this cell safely after any interruption.
    """
    print(f"\n{'='*64}")
    print(f"  Config : {cfg_name}")
    print(f"  Active : IQ={use_iq}  Const={use_const}  Spec={use_spec}")
    print(f"  Epochs : {NUM_EPOCHS}  |  Seeds : {NUM_RUNS}  |  Batch : {BATCH_SIZE}")
    print(f"{'='*64}")

    done_seeds = all_seeds_done(cfg_name)
    if done_seeds:
        print(f"  ↩  Already completed seeds: {done_seeds} — loading from disk …")

    all_train_losses, all_val_accs, all_best_accs, all_snr_accs = [], [], [], []

    for seed in SEEDS:
        path = ckpt_path(cfg_name, seed)

        # ── Resume: load existing checkpoint ───────────────────────────────
        if os.path.exists(path):
            ckpt = torch.load(path, map_location=device)
            all_train_losses.append(ckpt["train_losses"])
            all_val_accs.append(ckpt["val_accs"])
            all_best_accs.append(ckpt["best_acc"])
            all_snr_accs.append(ckpt["snr_acc"])
            print(f"  Seed {seed} — loaded  (best acc: {ckpt['best_acc']*100:.2f}%)")
            continue

        # ── Train from scratch ──────────────────────────────────────────────
        print(f"\n  ── Seed {seed+1}/{NUM_RUNS}  (seed={seed}) ──")
        set_seed(seed)
        model = FusionModel(num_classes=len(MOD_NAMES),
                            use_iq=use_iq, use_const=use_const,
                            use_spec=use_spec).to(device)

        t0 = time.time()
        train_losses, val_accs, best_state, snr_acc = train_one_run(model, NUM_EPOCHS, LR)
        elapsed = time.time() - t0
        best_acc = max(val_accs)

        all_train_losses.append(train_losses)
        all_val_accs.append(val_accs)
        all_best_accs.append(best_acc)
        all_snr_accs.append(snr_acc)

        torch.save({
            "config":       cfg_name,
            "use_iq":       use_iq,
            "use_const":    use_const,
            "use_spec":     use_spec,
            "seed":         seed,
            "num_epochs":   NUM_EPOCHS,
            "best_acc":     best_acc,
            "state_dict":   best_state,
            "train_losses": train_losses,
            "val_accs":     val_accs,
            "snr_acc":      snr_acc,
        }, path)

        print(f"     Best acc : {best_acc*100:.2f}%  |  Time : {elapsed/60:.1f} min")
        print(f"     Saved   → {path}")

    # ── Learning curves ─────────────────────────────────────────────────────
    plot_learning_curves(cfg_name, all_train_losses, all_val_accs, save=True)

    results[cfg_name] = {"accs": all_best_accs, "snr_accs": all_snr_accs}
    mean_acc = np.mean(all_best_accs) * 100
    std_acc  = np.std(all_best_accs)  * 100
    print(f"\n  ✓ {cfg_name}  →  {mean_acc:.2f}% ± {std_acc:.2f}%")
    return results[cfg_name]


## 7. Run Each Configuration

Each cell is self-contained and **auto-resumes** from saved checkpoints.  
If the session is interrupted, re-run cells 1–6 to reload helpers, then re-run
only the config cells that were not finished — completed seeds are skipped automatically.


### 7.1 — IQ only
Active branches: **IQ**

In [ ]:
run_config("IQ only", use_iq=True, use_const=False, use_spec=False)

### 7.2 — Const only
Active branches: **Constellation**

In [ ]:
run_config("Const only", use_iq=False, use_const=True, use_spec=False)

### 7.3 — Spec only
Active branches: **Spectrogram**

In [ ]:
run_config("Spec only", use_iq=False, use_const=False, use_spec=True)

### 7.4 — IQ + Const
Active branches: **IQ + Constellation**

In [ ]:
run_config("IQ + Const", use_iq=True, use_const=True, use_spec=False)

### 7.5 — IQ + Spec
Active branches: **IQ + Spectrogram**

In [ ]:
run_config("IQ + Spec", use_iq=True, use_const=False, use_spec=True)

### 7.6 — Const + Spec
Active branches: **Constellation + Spectrogram**

In [ ]:
run_config("Const + Spec", use_iq=False, use_const=True, use_spec=True)

### 7.7 — Full Fusion
Active branches: **IQ + Constellation + Spectrogram**

In [ ]:
run_config("Full Fusion", use_iq=True, use_const=True, use_spec=True)

## 8. Aggregate Results
> Run after all 7 configuration cells have completed.

In [ ]:
CFG_ORDER = [c[0] for c in [
    ("IQ only",      True,  False, False),
    ("Const only",   False, True,  False),
    ("Spec only",    False, False, True),
    ("IQ + Const",   True,  True,  False),
    ("IQ + Spec",    True,  False, True),
    ("Const + Spec", False, True,  True),
    ("Full Fusion",  True,  True,  True),
]]

summary = {}
print(f"{'Configuration':<18}  {'Mean':>8}  {'Std':>6}  {'Min':>6}  {'Max':>6}")
print("-" * 52)
for name in CFG_ORDER:
    if name not in results:
        print(f"{name:<18}  (not run yet)"); continue
    accs = results[name]["accs"]
    m, s = np.mean(accs)*100, np.std(accs)*100
    summary[name] = (m, s)
    print(f"{name:<18}  {m:>7.2f}%  {s:>5.2f}%  {min(accs)*100:>5.2f}%  {max(accs)*100:>5.2f}%")


In [ ]:
# ── Bar chart ─────────────────────────────────────────────────────────────────
names   = [n for n in CFG_ORDER if n in summary]
means   = [summary[n][0] for n in names]
stds    = [summary[n][1] for n in names]
colours = ["#7f7f7f"]*3 + ["#4e8ecb"]*3 + ["#e8a838"]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(names, means, yerr=stds, capsize=5,
              color=colours, edgecolor="black", linewidth=0.6,
              width=0.55, error_kw=dict(elinewidth=1.5, ecolor="black"))
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.3,
            f"{m:.1f}%", ha="center", va="bottom", fontsize=8.5, fontweight="bold")
ax.set_ylabel("Test Accuracy (%)", fontsize=11)
ax.set_title(f"Ablation Study — Mean ± Std Accuracy ({NUM_RUNS} seeds, {NUM_EPOCHS} epochs)", fontsize=12)
ax.set_ylim(0, min(105, max(means) + max(stds) + 8))
ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.set_xticklabels(names, rotation=15, ha="right")
ax.legend(handles=[
    mpatches.Patch(facecolor="#7f7f7f", edgecolor="black", label="Single branch"),
    mpatches.Patch(facecolor="#4e8ecb", edgecolor="black", label="Two branches"),
    mpatches.Patch(facecolor="#e8a838", edgecolor="black", label="Full fusion"),
], loc="lower right")
plt.tight_layout()
save_fig(fig, "ablation_bar_chart.png")
plt.show()


In [ ]:
# ── Per-SNR accuracy curves ────────────────────────────────────────────────────
snrs_unique = sorted(np.unique(snr_test))
palette     = plt.cm.tab10(np.linspace(0, 1, len(CFG_ORDER)))

fig, ax = plt.subplots(figsize=(11, 6))
for name, colour in zip(CFG_ORDER, palette):
    if name not in results: continue
    runs   = results[name]["snr_accs"]
    mean_c = np.array([np.mean([r[s] for r in runs]) for s in snrs_unique])
    std_c  = np.array([np.std( [r[s] for r in runs]) for s in snrs_unique])
    ls     = "-" if "IQ" in name else "--"
    ax.plot(snrs_unique, mean_c*100, label=name, color=colour,
            linewidth=2, linestyle=ls, marker="o", markersize=4)
    ax.fill_between(snrs_unique, (mean_c-std_c)*100, (mean_c+std_c)*100,
                    alpha=0.12, color=colour)
ax.axhline(20, color="black", linestyle=":", linewidth=1, alpha=0.5, label="Random (20%)")
ax.set_xlabel("SNR (dB)", fontsize=11); ax.set_ylabel("Accuracy (%)", fontsize=11)
ax.set_title(f"Per-SNR Accuracy — All Configurations\n(shaded = ±1 std, {NUM_RUNS} seeds)", fontsize=12)
ax.legend(loc="upper left", fontsize=8.5, ncol=2)
ax.grid(True, linestyle="--", alpha=0.4); ax.set_xticks(snrs_unique)
plt.tight_layout()
save_fig(fig, "per_snr_accuracy.png")
plt.show()


In [ ]:
# ── Heat-map ──────────────────────────────────────────────────────────────────
snrs_unique  = sorted(np.unique(snr_test))
active_names = [n for n in CFG_ORDER if n in results]

heatmap = np.array([[np.mean([r[snr] for r in results[name]["snr_accs"]])
                     for snr in snrs_unique] for name in active_names])

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(heatmap*100, aspect="auto", cmap="RdYlGn", vmin=0, vmax=100)
fig.colorbar(im, ax=ax, pad=0.02).set_label("Accuracy (%)")
ax.set_xticks(range(len(snrs_unique))); ax.set_xticklabels(snrs_unique, fontsize=8)
ax.set_yticks(range(len(active_names))); ax.set_yticklabels(active_names, fontsize=9)
ax.set_xlabel("SNR (dB)")
ax.set_title(f"Mean Accuracy Heat-map — Configuration × SNR  ({NUM_RUNS} seeds)")
for ci in range(len(active_names)):
    for si in range(len(snrs_unique)):
        v = heatmap[ci, si]*100
        ax.text(si, ci, f"{v:.0f}", ha="center", va="center",
                fontsize=6.5, color="black" if 20 < v < 80 else "white")
plt.tight_layout()
save_fig(fig, "accuracy_heatmap.png")
plt.show()


## 9. Statistical Significance Testing

In [ ]:
baseline = "Full Fusion"
if baseline in results:
    base_accs = results[baseline]["accs"]
    print(f"Pairwise t-tests vs '{baseline}'  (two-sided, α = 0.05)\n")
    print(f"{'Config':<18}  {'Δ Acc':>8}  {'p-value':>10}  {'Significant?':>14}")
    print("-" * 56)
    for name in CFG_ORDER:
        if name == baseline or name not in results: continue
        _, p  = ttest_ind(base_accs, results[name]["accs"])
        delta = (np.mean(base_accs) - np.mean(results[name]["accs"])) * 100
        print(f"{name:<18}  {delta:>+7.2f}%  {p:>10.4f}  {'YES ✓' if p < 0.05 else 'no':>14}")
else:
    print("Full Fusion not run yet.")


## 10. Export Results

Run this after all configs complete to generate files for the GitHub repository.

In [ ]:
# ── Summary CSV ───────────────────────────────────────────────────────────────
csv_path = "results/results_summary.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Configuration","Mean_Acc","Std_Acc","Min_Acc","Max_Acc","Num_Seeds","Num_Epochs"])
    for name in CFG_ORDER:
        if name not in results: continue
        accs = results[name]["accs"]
        w.writerow([name, f"{np.mean(accs)*100:.4f}", f"{np.std(accs)*100:.4f}",
                    f"{min(accs)*100:.4f}", f"{max(accs)*100:.4f}", NUM_RUNS, NUM_EPOCHS])
print(f"Saved → {csv_path}")

# ── Per-SNR CSV ───────────────────────────────────────────────────────────────
snrs_unique  = sorted(np.unique(snr_test))
snr_csv_path = "results/per_snr_accuracy.csv"
with open(snr_csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Configuration","SNR_dB","Mean_Acc","Std_Acc"])
    for name in CFG_ORDER:
        if name not in results: continue
        for snr in snrs_unique:
            vals = [r[snr] for r in results[name]["snr_accs"]]
            w.writerow([name, snr, f"{np.mean(vals)*100:.4f}", f"{np.std(vals)*100:.4f}"])
print(f"Saved → {snr_csv_path}")

# ── Full JSON ─────────────────────────────────────────────────────────────────
json_path = "results/results_full.json"
export = {name: {"accs": results[name]["accs"],
                 "snr_accs": [{int(k): v for k,v in r.items()}
                              for r in results[name]["snr_accs"]],
                 "num_seeds": NUM_RUNS, "num_epochs": NUM_EPOCHS}
          for name in CFG_ORDER if name in results}
with open(json_path, "w") as f:
    json.dump(export, f, indent=2)
print(f"Saved → {json_path}")

print("\n✓ All results exported. Download from the Output tab on the right.")
print("\nOutput contents:")
for root, dirs, files in os.walk("."):
    for file in sorted(files):
        p = os.path.join(root, file)
        print(f"  {p}  ({os.path.getsize(p)/1024:.0f} KB)")


## 11. Load Saved Models

In [ ]:
def load_checkpoint(cfg_name, seed):
    """Reload a saved checkpoint for inference or further analysis."""
    path = ckpt_path(cfg_name, seed)
    ckpt = torch.load(path, map_location=device)
    model = FusionModel(num_classes=len(MOD_NAMES),
                        use_iq=ckpt["use_iq"],
                        use_const=ckpt["use_const"],
                        use_spec=ckpt["use_spec"]).to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    print(f"Loaded  : {path}")
    print(f"Config  : {ckpt['config']}  |  Seed : {ckpt['seed']}  |  Epochs : {ckpt['num_epochs']}")
    print(f"Best acc: {ckpt['best_acc']*100:.2f}%")
    return model, ckpt

# ── Example ───────────────────────────────────────────────────────────────────
# model, ckpt = load_checkpoint("Full Fusion", seed=0)
# plot_learning_curves(ckpt["config"], [ckpt["train_losses"]], [ckpt["val_accs"]], save=False)

print("load_checkpoint() ready.")
print("\nExisting checkpoints:")
if os.path.exists("saved_models"):
    for f in sorted(os.listdir("saved_models")):
        p = os.path.join("saved_models", f)
        print(f"  {f:<45}  {os.path.getsize(p)/1024:>6.0f} KB")
else:
    print("  None yet — run the config cells first.")
